# 数据管理与组织

运行完 `codes_get_data.ipynb` 之后，你的 `data_raw/` 文件夹里已经有了 17 个文件：10 只个股的日度行情、3 个市场指数、3 个宏观指标、1 份年度财务表、1 份公司信息表，合计超过 2 万行数据。

现在问题来了。假设你要做这样一个分析：找出 2023 年以来，制造业上市公司中换手率最高的 10 个交易日，同时看看这些日期对应的 Shibor 利率是多少。用现有的文件怎么做？你需要先循环读入 10 个 CSV，`concat` 成一张大表，再读入公司信息表做 `merge`，再读入 Shibor 数据按月对齐……每次分析都要重复这套流程，既慢，也容易出错。

这就是本章要解决的问题：**数据多了之后，如何管好它们。**

本章的主线是：

$$\text{原始文件} \xrightarrow{\text{组织}} \text{数据库/Parquet} \xrightarrow{\text{查询}} \text{分析底表} \xrightarrow{\text{清洗建模}} \text{结论}$$

这不是一个非此即彼的选择——pandas 仍然是核心工具，我们只是在它的上游加一层更好的数据组织方式。

## 本章导读

本章的目标不是教你记住数据库的所有术语，而是帮你建立三种意识：

- **粒度意识**：每张表的每一行代表什么？是一个公司，还是一个公司-年，还是一个公司-交易日？
- **主键意识**：什么变量组合能唯一识别一行数据？主键不对，`merge` 就会出问题。
- **复用意识**：数据组织好了，每次分析只写查询，不重复读文件、不重复 `merge`，代码更短，也更不容易出错。

本章使用的数据集如下：

| 数据集 | 文件 | 粒度 | 主键 | 行数 |
|--------|------|------|------|------|
| 个股日度行情 | `stock_daily/*.csv` | firm-date | `code + date` | 15,100 |
| 市场指数日度 | `index_daily/*.csv` | index-date | `index_code + date` | 4,533 |
| 宏观月度 | `macro_monthly/*.csv` | date（月）| `date` | 223 |
| 年度财务指标 | `fin_annual/fin_indicators.csv` | firm-year | `code + year` | 60 |
| 公司基本信息 | `company_info.csv` | firm | `code` | 10 |

&emsp; 

| 节次 | 内容 | 核心工具 |
|------|------|----------|
| Sec 1 | 数据表、主键与粒度 | `pandas` |
| Sec 2 | 存储格式的选择 | CSV / Parquet / SQLite / DuckDB |
| Sec 3 | 用 SQLite 管理金融数据 | `sqlite3` + `pandas` |
| Sec 4 | SQL 核心操作 | SQL + `pd.read_sql()` |
| Sec 5 | 大数据量：DuckDB + Parquet | `duckdb` |
| Sec 6 | 数据管理规范 | `pathlib` |

---

## Sec 1  数据表、主键与粒度

在动手建库之前，先把三个基础概念弄清楚。它们不是数据库理论，而是做好任何数据分析都需要的思维框架。

### 粒度

**粒度**就是「每一行代表什么」。金融数据中最常见的几种：

| 粒度 | 含义 | 典型数据 |
|------|------|----------|
| `firm` | 每行是一家公司 | 公司基本信息 |
| `firm-year` | 每行是某公司某年 | 年度财务报表 |
| `firm-date` | 每行是某公司某交易日 | 日度行情 |
| `date` | 每行是某一天或某月 | 宏观指标、指数 |

### 主键

**主键**是能唯一识别一行数据的变量或变量组合。若 $K$ 是主键，则对任意两行 $i \neq j$，必有 $K_i \neq K_j$。

- `company_info` 的主键是 `code`
- `fin_annual` 的主键是 `(code, year)`
- `stock_daily` 的主键是 `(code, date)`

**merge 之前先验证主键**，是避免数据问题的第一道防线。

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path('data_raw')

# ── Mini-dataset：3只股票，字段名与 data_raw/ 完全一致 ──
# 后面切换到真实数据时代码不用改

company_mini = pd.DataFrame({
    'code'       : ['000001', '600519', '300750'],
    'name'       : ['平安银行', '贵州茅台', '宁德时代'],
    'industry_l1': ['金融',    '消费',    '制造'],
    'ownership'  : ['国企',    '民企',    '民企'],
})

fin_mini = pd.DataFrame({
    'code'  : ['000001','000001','600519','600519','300750','300750'],
    'year'  : [2022,    2023,    2022,    2023,    2022,    2023   ],
    'pe_ttm': [5.2,     5.8,     28.4,    24.1,    35.6,    22.3  ],
    'pb'    : [0.6,     0.6,     9.1,     8.4,     4.2,     3.1   ],
})

daily_mini = pd.DataFrame({
    'code'    : ['000001']*3 + ['600519']*3,
    'date'    : ['2024-01-02','2024-01-03','2024-01-04'] * 2,
    'close'   : [10.2, 10.5, 10.3, 1680.0, 1695.0, 1702.0],
    'pct_chg' : [-0.5,  2.9, -1.9,    0.3,    0.9,    0.4 ],
    'turnover': [ 0.8,  1.2,  0.9,    0.2,    0.3,    0.2 ],
})
daily_mini['date'] = pd.to_datetime(daily_mini['date'])

print('三张表的粒度一览：')
print(f'  company_mini：{len(company_mini)} 行（每行 = 一家公司）')
print(f'  fin_mini    ：{len(fin_mini)} 行（每行 = 一家公司 × 一年）')
print(f'  daily_mini  ：{len(daily_mini)} 行（每行 = 一家公司 × 一个交易日）')

In [ ]:
# ── 验证主键唯一性：merge 之前必须做这个检查 ───────────

checks = [
    ('company_mini', company_mini, ['code']),
    ('fin_mini',     fin_mini,     ['code', 'year']),
    ('daily_mini',   daily_mini,   ['code', 'date']),
]

for name, df, keys in checks:
    has_dup = df.duplicated(subset=keys).any()
    status  = '✗ 有重复！' if has_dup else '✓ 主键唯一'
    print(f'{name:<15}  主键：{keys}  →  {status}')

### 粒度错误：最隐蔽的 merge 陷阱

`fin_mini` 是 firm-year 粒度，`daily_mini` 是 firm-date 粒度。如果只按 `code` 做 merge，会发生什么？

In [ ]:
# ── 错误做法：只按 code 合并不同粒度的表 ──────────────
wrong = fin_mini.merge(daily_mini, on='code', how='left')

print(f'fin_mini    原始行数：{len(fin_mini)}')
print(f'daily_mini  原始行数：{len(daily_mini)}')
print(f'合并后行数：         {len(wrong)}   ← 莫名增加了！')
print()
print(wrong)

原本 6 行的财务表，merge 之后变成了 18 行。原因是：平安银行在 `fin_mini` 里有 2 行（两年），在 `daily_mini` 里有 3 行（三个交易日），合并后产生了 2×3=6 行。每年的财务数据都和每个交易日配了一遍，没有任何经济含义。代码不会报错，结果也看起来「正常」——这才是它危险的地方。

**根本原因不是代码写错了，而是粒度没对齐。** 正确做法是先把高频数据聚合到目标粒度：

In [ ]:
# ── 正确做法：先把日度数据聚合到 firm-year，再 merge ──

# 第一步：从日期提取年份
daily_agg = daily_mini.copy()
daily_agg['year'] = daily_agg['date'].dt.year

# 第二步：按 code+year 聚合，计算年度统计量
daily_agg = (
    daily_agg
    .groupby(['code', 'year'], as_index=False)
    .agg(
        avg_ret      = ('pct_chg',  'mean'),
        volatility   = ('pct_chg',  'std'),
        avg_turnover = ('turnover', 'mean'),
    )
)

# 第三步：两张表粒度一致（firm-year），可以安全 merge
result = fin_mini.merge(daily_agg, on=['code', 'year'], how='left')

print(f'合并后行数：{len(result)}（与 fin_mini 原始行数一致，没有膨胀）')
print()
print(result.round(3))

这个模式在后面的真实数据里会反复用到：**日度数据 → 年度聚合 → 与财务表 merge**，是金融实证研究中最典型的数据整合流程之一。

::: {.callout-tip}
### AI 提示词：检查粒度与主键

我有两张 pandas DataFrame：

- `df_annual`（字段：code, year, roe, leverage，粒度为 firm-year）
- `df_daily`（字段：code, date, pct_chg, volume，粒度为 firm-date）

我想构造一张 firm-year 分析底表，把日度收益率的年均值和年波动率并入财务数据。请帮我：  
(1) 先验证两张表的主键是否唯一；  
(2) 把日度数据聚合为年度；  
(3) 按 (code, year) 合并，用 left join；  
(4) 输出合并后的行数，确认没有行数膨胀。
:::

---

## Sec 2  存储格式的选择

拿到数据、理清了粒度之后，下一个问题是：用什么格式存？这取决于你的使用场景，没有唯一正确答案。

| 格式 | 最适合 | 不适合 | 关键特点 |
|------|--------|--------|----------|
| CSV | 分享、人工查看、小数据 | 大文件、频繁读写 | 无类型信息，通用性最强 |
| Parquet | 大表、重复读取、列查询 | 人工查看 | 列式存储，压缩好，速度快 |
| SQLite | 多表关联、反复查询 | 高并发写入 | 单文件数据库，支持 SQL |
| DuckDB | 超大量分析查询 | 事务型写入 | 可直接 SQL 查 Parquet |

下面用平安银行的数据做一个直观对比。

In [ ]:
import timeit

csv_path     = BASE / 'stock_daily' / '000001.csv'
parquet_path = BASE / 'stock_daily' / '000001.parquet'

# 先把 CSV 转存为 Parquet
df_sample = pd.read_csv(csv_path)
df_sample.to_parquet(parquet_path, index=False)

# ── 文件大小对比 ──────────────────────────────────────
csv_kb = csv_path.stat().st_size / 1024
pq_kb  = parquet_path.stat().st_size / 1024
print(f'文件大小对比（平安银行 {len(df_sample):,} 行）：')
print(f'  CSV     ：{csv_kb:.1f} KB')
print(f'  Parquet ：{pq_kb:.1f} KB')
print(f'  压缩比  ：{csv_kb / pq_kb:.1f}x')

# ── 读取速度对比（各测 30 次取均值）──────────────────
t_csv = timeit.timeit(lambda: pd.read_csv(csv_path),         number=30) / 30
t_pq  = timeit.timeit(lambda: pd.read_parquet(parquet_path), number=30) / 30

print(f'\n读取速度对比（30次均值）：')
print(f'  CSV     ：{t_csv*1000:.1f} ms')
print(f'  Parquet ：{t_pq*1000:.1f} ms')
print(f'  速度提升：{t_csv / t_pq:.1f}x')

单只股票 1510 行的数据量还不大，差距不算惊人。Parquet 的优势在数据量达到几十万行以上时才会真正体现——届时读取时间差距可达 5-10 倍，文件体积差距也更显著。此外，Parquet 还会**保留数据类型**：CSV 里日期列读进来默认是字符串，Parquet 会记住「这列是 datetime」，读进来就是正确的类型，省去每次手动转换。

::: {.callout-note}
### 什么时候应该用 Parquet 而不是 CSV？

- 这份数据会被反复读取（每次分析都要读一遍）
- 文件超过 10MB
- 你只需要其中几列，不需要每次读全部字段
- 需要保留日期、布尔值等类型信息

如果只是偶尔查看，或者需要发给别人、上传到系统，CSV 依然是最方便的选择。
:::

---

## Sec 3  用 SQLite 管理金融数据

SQLite 是 Python 标准库内置的关系型数据库，不需要安装服务器、不需要配置账号密码，整个数据库就是一个 `.db` 文件。对于分析师来说，它解决的核心问题是：**把多张表统一放在一个地方，用 SQL 查询替代反复的 `read_csv` + `merge`。**

这一节先用 mini-dataset 建一个内存数据库，把基本操作看清楚；再把真实的 17 个文件导入一个持久化数据库。

### Part A：用 mini-dataset 理解建库操作

In [ ]:
import sqlite3

# ':memory:' 表示在内存中建库，不写磁盘，适合演示和测试
# 真实项目用文件路径，如 sqlite3.connect('data_raw/finance.db')
conn_mini = sqlite3.connect(':memory:')

# pandas 的 to_sql() 直接把 DataFrame 写入数据库表
company_mini.to_sql('company',    conn_mini, if_exists='replace', index=False)
fin_mini.to_sql(    'fin_annual',  conn_mini, if_exists='replace', index=False)
daily_mini.to_sql(  'stock_daily', conn_mini, if_exists='replace', index=False)

# 查看数据库中有哪些表
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn_mini
)
print('数据库中的表：')
print(tables)

In [ ]:
# ── SELECT + WHERE + ORDER BY ──────────────────────────
# pd.read_sql() 直接把查询结果返回为 DataFrame

query = '''
SELECT code, year, pe_ttm, pb
FROM   fin_annual
WHERE  pe_ttm < 10
ORDER  BY pe_ttm ASC
'''
pd.read_sql(query, conn_mini)

In [ ]:
# ── GROUP BY + JOIN ─────────────────────────────────────
# 按行业计算平均 pe_ttm（先 join 公司信息获取行业字段）

query = '''
SELECT   c.industry_l1,
         COUNT(*)               AS n,
         ROUND(AVG(f.pe_ttm), 2) AS avg_pe
FROM     fin_annual f
JOIN     company    c  ON f.code = c.code
WHERE    f.year = 2023
GROUP BY c.industry_l1
ORDER BY avg_pe
'''
pd.read_sql(query, conn_mini)

以上三行 SQL 对应的 pandas 写法：

```python
merged = fin_mini.merge(company_mini, on='code')
merged[merged['year']==2023].groupby('industry_l1')['pe_ttm'].agg(['count','mean'])
```

两种写法都能得到同样的结果。当数据已经在数据库里、需要多表关联时，SQL 的表达往往更直接。

### Part B：真实数据建库

In [ ]:
import glob

DB_PATH = BASE / 'finance.db'
conn = sqlite3.connect(DB_PATH)

# ── 1. 个股日度（10 个 CSV 合并写入一张表）────────────
files    = sorted(glob.glob(str(BASE / 'stock_daily/*.csv')))
df_stock = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_stock.to_sql('stock_daily', conn, if_exists='replace', index=False)
print(f'stock_daily 写入完成：{len(df_stock):,} 行')

# ── 2. 市场指数 ──────────────────────────────────────
files  = sorted(glob.glob(str(BASE / 'index_daily/*.csv')))
df_idx = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_idx.to_sql('index_daily', conn, if_exists='replace', index=False)
print(f'index_daily 写入完成：{len(df_idx):,} 行')

# ── 3. 宏观月度（文件名作为表名）────────────────────
for f in sorted(glob.glob(str(BASE / 'macro_monthly/*.csv'))):
    tbl = Path(f).stem
    df  = pd.read_csv(f)
    df.to_sql(tbl, conn, if_exists='replace', index=False)
    print(f'{tbl} 写入完成：{len(df)} 行')

# ── 4. 年度财务 ──────────────────────────────────────
df_fin = pd.read_csv(BASE / 'fin_annual' / 'fin_indicators.csv')
df_fin.to_sql('fin_annual', conn, if_exists='replace', index=False)
print(f'fin_annual 写入完成：{len(df_fin)} 行')

# ── 5. 公司信息（处理 key-value 长格式）─────────────
df_co_raw = pd.read_csv(BASE / 'company_info.csv')
if len(df_co_raw) > 20:   # 是长格式（160行），需要转换为宽格式
    key_map  = {'行业': 'industry_l1', '地区': 'province',
                '上市时间': 'list_date', '实控人': 'controller'}
    col0, col1 = df_co_raw.columns[0], df_co_raw.columns[1]
    records  = []
    code_col = [c for c in df_co_raw.columns if 'code' in c.lower()]
    if code_col:
        for code_val in df_co_raw[code_col[0]].unique():
            sub = df_co_raw[df_co_raw[code_col[0]] == code_val]
            row = {'code': code_val}
            for _, r in sub.iterrows():
                k = str(r[col0])
                if k in key_map:
                    row[key_map[k]] = r[col1]
            if 'controller' in row:
                row['ownership'] = '国企' if any(
                    kw in str(row['controller'])
                    for kw in ['国资','国有','国家','政府']
                ) else '民企'
            records.append(row)
        df_co = pd.DataFrame(records)
    else:
        df_co = df_co_raw.head(10)
else:
    df_co = df_co_raw

# 补充股票名称
names = df_stock[['code','name']].drop_duplicates('code')
df_co = df_co.merge(names, on='code', how='left')
df_co.to_sql('company_info', conn, if_exists='replace', index=False)
print(f'company_info 写入完成：{len(df_co)} 行')
print(f'\n数据库已保存至 {DB_PATH}')

In [ ]:
# ── 验证：查看各表行数 ──────────────────────────────
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
print(f'数据库大小：{DB_PATH.stat().st_size / 1024 / 1024:.1f} MB\n')
print(f'{"表名":<20} {"行数":>8}')
print('-' * 30)
for tbl in tables['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as n FROM {tbl}', conn)['n'][0]
    print(f'{tbl:<20} {n:>8,}')

17 个散落的 CSV 文件，现在都在一个 `.db` 文件里，可以随时用 SQL 查询、关联。

**与「每次 concat 再 merge」相比，建库的价值在于：**
第一次建库需要几秒钟；之后每次分析只需要写 SQL，不再循环读文件。查询时可以只取需要的行和列，不把全量数据读进内存。

::: {.callout-tip}
### AI 提示词：把多个 CSV 导入 SQLite

> 我有一个文件夹 `data_raw/stock_daily/`，里面有 10 个 CSV，字段包括 date, open, close, high, low, volume, amount, pct_chg, turnover, code, name。请帮我写 Python 代码，把这 10 个文件合并后写入 SQLite 数据库 `data_raw/finance.db` 的 `stock_daily` 表，写入后验证总行数和时间范围，并用 try/except 处理读取错误。
:::

---

## Sec 4  SQL 核心操作

SQL 不是新技能，是你已经掌握的 pandas 操作的另一种表达方式。

| 操作 | pandas 写法 | SQL 写法 |
|------|------------|----------|
| 选列 | `df[['code','close']]` | `SELECT code, close` |
| 筛选行 | `df[df['pct_chg'] > 5]` | `WHERE pct_chg > 5` |
| 排序 | `df.sort_values('close', ascending=False)` | `ORDER BY close DESC` |
| 取前 N 行 | `df.nlargest(10, 'volume')` | `ORDER BY volume DESC LIMIT 10` |
| 分组均值 | `df.groupby('code')['pct_chg'].mean()` | `GROUP BY code` + `AVG(pct_chg)` |
| 多表合并 | `pd.merge(df1, df2, on='code')` | `JOIN ... ON code` |
| 新建列 | `df['x'] = df['pct_chg'] * 252` | `pct_chg * 252 AS x` |
| 提取年份 | `df['date'].dt.year` | `STRFTIME('%Y', date)` |

下面用真实数据演示四个有实际分析意义的查询。

In [ ]:
# 如果重新打开 notebook，需要重新连接数据库
import sqlite3, pandas as pd
from pathlib import Path
BASE = Path('data_raw')
conn = sqlite3.connect(BASE / 'finance.db')

In [ ]:
# ── 查询 1：各股票 2024 年表现概览 ─────────────────────
# 计算年化收益率、年化波动率、平均换手率，关联公司信息

q1 = '''
SELECT
    s.code,
    c.name,
    c.industry_l1,
    c.ownership,
    ROUND(AVG(s.pct_chg), 3)                   AS avg_daily_ret,
    ROUND(AVG(s.pct_chg) * 252, 1)             AS ret_annualized,
    ROUND(AVG(ABS(s.pct_chg)) * 15.87, 2)      AS vol_annualized,
    ROUND(AVG(s.turnover), 3)                  AS avg_turnover
FROM   stock_daily  s
JOIN   company_info c  ON s.code = c.code
WHERE  s.date BETWEEN '2024-01-01' AND '2024-12-31'
GROUP  BY s.code
ORDER  BY ret_annualized DESC
'''

df_q1 = pd.read_sql(q1, conn)
print(f'查询结果：{len(df_q1)} 只股票')
df_q1

In [ ]:
# ── 查询 2：找出异常成交日（成交量超过年均 3 倍）───────
# 异常成交日往往对应重大事件：业绩披露、政策出台、指数调仓等

q2 = '''
SELECT
    s.code,
    c.name,
    s.date,
    ROUND(s.volume / 10000, 2)  AS volume_wan,
    ROUND(s.pct_chg, 2)         AS pct_chg,
    ROUND(s.turnover, 2)        AS turnover
FROM   stock_daily  s
JOIN   company_info c  ON s.code = c.code
JOIN (
    SELECT code, AVG(volume) AS avg_vol
    FROM   stock_daily
    WHERE  date BETWEEN '2024-01-01' AND '2024-12-31'
    GROUP  BY code
) avg ON s.code = avg.code
WHERE  s.date BETWEEN '2024-01-01' AND '2024-12-31'
  AND  s.volume > avg.avg_vol * 3
ORDER  BY s.volume DESC
LIMIT  20
'''

df_q2 = pd.read_sql(q2, conn)
print(f'2024 年成交量超过年均 3 倍的交易日：{len(df_q2)} 条')
df_q2

In [ ]:
# ── 查询 3：混频 JOIN——日度行情 × 月度 Shibor ──────────
# 把每个交易日对应到所在月份的 Shibor 利率
# 关键：用 STRFTIME 把日度日期截取为 YYYY-MM，与月度数据的 date 列做匹配

q3 = '''
SELECT
    s.date,
    s.code,
    c.name,
    ROUND(s.pct_chg, 2)            AS pct_chg,
    r.shibor_3m,
    STRFTIME('%Y-%m', s.date)       AS ym
FROM   stock_daily  s
JOIN   company_info c   ON s.code = c.code
LEFT JOIN shibor_3m r   ON STRFTIME('%Y-%m', s.date) = r.date
WHERE  s.code = '000001'
  AND  s.date >= '2024-01-01'
ORDER  BY s.date
LIMIT  20
'''

df_q3 = pd.read_sql(q3, conn)
print('平安银行日度收益率 + 当月 Shibor（前 20 行）：')
df_q3

查询 3 展示了混频数据连接的标准做法：用 `STRFTIME('%Y-%m', date)` 把日度日期截取为月份字符串，与月度数据的 `date` 列（格式 `YYYY-MM`）做 JOIN。每个交易日都能找到对应月份的 Shibor，实现「月度数据广播到每个交易日」。

等价的 pandas 写法要繁琐得多：先提取月份列、再 merge、再处理键格式不一致……这正是 SQL 擅长的场景。

In [ ]:
# ── 查询 4：日度数据聚合到年度，与财务表合并 ───────────
# 金融实证最常见的数据整合模式：日度行情 → 年度风险指标 → 并入财务数据

q4 = '''
SELECT
    f.code,
    c.name,
    c.industry_l1,
    f.year,
    ROUND(f.pe_ttm, 2)                         AS pe_ttm,
    ROUND(f.pb, 2)                             AS pb,
    ROUND(r.avg_ret_daily * 252 * 100, 2)      AS ret_ann_pct,
    COUNT(r.code)                              AS trading_days
FROM fin_annual f
JOIN company_info c ON f.code = c.code
JOIN (
    SELECT
        code,
        CAST(STRFTIME('%Y', date) AS INTEGER)  AS year,
        AVG(pct_chg / 100)                     AS avg_ret_daily
    FROM  stock_daily
    GROUP BY code, STRFTIME('%Y', date)
) r ON f.code = r.code AND f.year = r.year
ORDER BY f.year, ret_ann_pct DESC
'''

df_q4 = pd.read_sql(q4, conn)
print(f'firm-year 分析底表：{len(df_q4)} 行')
df_q4.head(15)

### 什么时候用 SQL，什么时候用 pandas？

**优先用 SQL：** 数据在数据库里需要筛选聚合；需要多表关联；混频数据连接；只需要部分行列。

**优先用 pandas：** 需要复杂向量化计算（滚动均值、信号生成）；数据已经在 DataFrame 里；绘图可视化。

实际项目中最常见的模式是：**SQL 负责数据整合和过滤，pandas 负责最终计算和可视化。**

::: {.callout-tip}
### AI 提示词：SQL 多表关联分析

> 我有一个 SQLite 数据库（`data_raw/finance.db`），包含 `stock_daily`（code, date, close, pct_chg, volume, turnover）、`company_info`（code, name, industry_l1, ownership）、`fin_annual`（code, year, pe_ttm, pb）、`shibor_3m`（date 格式 YYYY-MM, shibor_3m）。请帮我写一个 SQL 查询：找出 2023-2024 年间，国有企业中 pe_ttm 连续两年下降且年均日换手率超过 1% 的股票，输出股票代码、名称、两年的 pe_ttm 和平均换手率。用 `pd.read_sql()` 执行。
:::

---

## Sec 5  大数据量：DuckDB + Parquet

本章的数据量（约 2 万行）对 SQLite 来说小菜一碟。但在实际工作中，你可能会遇到更大规模的数据：全市场 5000 只股票 × 5 年日度数据约 900 万行，或者逐笔成交数据动辄数亿行。这时候 SQLite 的速度就不够用了。

**DuckDB** 是专为分析场景设计的嵌入式数据库，有两个特别适合金融数据分析的特点：
1. **直接查 Parquet 文件**，不需要先导入，`FROM 'file.parquet'` 就可以查
2. **列式存储引擎**，同样的聚合查询比 SQLite 快 10-50 倍（数据量越大差距越明显）

In [ ]:
import duckdb
import timeit

PARQUET_PATH = BASE / 'stock_daily' / 'all_stocks.parquet'

# 把 10 只股票合并保存为单个 Parquet 文件
if not PARQUET_PATH.exists():
    files  = sorted(glob.glob(str(BASE / 'stock_daily/*.csv')))
    df_all = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df_all.to_parquet(PARQUET_PATH, index=False)
    print(f'已保存：{PARQUET_PATH.stat().st_size/1024:.0f} KB')
else:
    print(f'文件已存在：{PARQUET_PATH.stat().st_size/1024:.0f} KB')

# DuckDB 直接查 Parquet，不需要任何导入步骤
result = duckdb.query(f"""
    SELECT code, COUNT(*) AS days
    FROM   '{PARQUET_PATH}'
    GROUP  BY code
    ORDER  BY code
""").df()

print('各股票交易日数：')
print(result.to_string(index=False))

In [ ]:
# ── 性能基准测试 ─────────────────────────────────────
# 任务：计算每只股票 2024 年的平均日收益率
# 三种方式做同一件事，看时间差距

N        = 20
csv_files = sorted(glob.glob(str(BASE / 'stock_daily/*.csv')))

def method_csv():
    dfs = [pd.read_csv(f) for f in csv_files]
    df  = pd.concat(dfs, ignore_index=True)
    return df[df['date'].str.startswith('2024')].groupby('code')['pct_chg'].mean()

def method_sqlite():
    return pd.read_sql(
        "SELECT code, AVG(pct_chg) AS avg_ret "
        "FROM stock_daily WHERE date LIKE '2024%' GROUP BY code",
        conn
    )

def method_duckdb():
    return duckdb.query(f"""
        SELECT code, AVG(pct_chg) AS avg_ret
        FROM   '{PARQUET_PATH}'
        WHERE  date LIKE '2024%'
        GROUP  BY code
    """).df()

t1 = timeit.timeit(method_csv,    number=N) / N * 1000
t2 = timeit.timeit(method_sqlite, number=N) / N * 1000
t3 = timeit.timeit(method_duckdb, number=N) / N * 1000

print(f'性能对比（{N}次均值，任务：10只股票 × 2024年均收益率）：')
print(f'  方法1  循环读 CSV + pandas  {t1:6.1f} ms')
print(f'  方法2  SQLite               {t2:6.1f} ms')
print(f'  方法3  DuckDB + Parquet     {t3:6.1f} ms')
print(f'\n  SQLite 比 CSV 快 {t1/t2:.1f}x')
print(f'  DuckDB 比 CSV 快 {t1/t3:.1f}x')
print('\n注：当数据量达到百万行以上时，DuckDB 的优势会更加显著')

在约 1.5 万行的数据量下，三种方法的差距有限。DuckDB 的真正价值在于更大的数据规模——当你面对全市场 5000 只股票的日度数据时，方法 1 可能需要几分钟，DuckDB 仍然在秒级以内完成。

::: {.callout-note}
### 三种方法的适用场景

| 方法 | 适合 | 不适合 |
|------|------|--------|
| 循环读 CSV + pandas | 文件少（<5个）、一次性分析 | 文件多、反复查询 |
| SQLite | 多表关联、中等数据量、教学 | 超大数据量（>千万行）|
| DuckDB + Parquet | 大数据量分析查询 | 频繁小事务写入 |

进入券商、基金等机构后，你会接触到 ClickHouse、Greenplum 等更专业的分析型数据库，使用逻辑与 DuckDB 相似——先写 SQL，再接 pandas 做最后处理。本章的 SQL 基础在那些场景下同样适用。
:::

::: {.callout-tip}
### AI 提示词：DuckDB 查询 Parquet

> 我有一个 Parquet 文件（`data_raw/stock_daily/all_stocks.parquet`），字段包括 code, date, close, pct_chg, volume, turnover。请帮我用 DuckDB 计算每只股票 2023-2024 年的年化收益率（日均收益率 × 252）和年化波动率（日收益率标准差 × √252），用 `duckdb.query(...).df()` 返回 DataFrame，按年化收益率降序排列。
:::

---

## Sec 6  数据管理规范

技术工具之外，数据项目能不能长期维护，很大程度上取决于一些非技术的习惯：目录怎么组织、文件怎么命名、数据来源怎么记录。这一节的内容不难，但养成习惯之后，能省去很多「这个文件是干什么的」「这份数据是什么时候下载的」之类的困惑。

### 目录结构

本章使用的目录结构可以作为金融数据分析项目的通用模板：

```
project/
├── data_raw/              ← 原始数据，只写入、不修改
│   ├── stock_daily/
│   ├── index_daily/
│   ├── macro_monthly/
│   ├── fin_annual/
│   ├── company_info.csv
│   ├── finance.db         ← 数据库文件（由代码生成，可重新生成）
│   └── download_log.txt
├── data_clean/            ← 清洗后的分析底表
├── output/                ← 图表、回归结果
├── codes_get_data.ipynb
├── data_store_manage.ipynb
└── analysis.ipynb
```

**最重要的一条：`data_raw/` 只写入、不修改。** 原始数据一旦下载，所有处理都保存到 `data_clean/`，保证随时可以从原始数据重新开始。

### 文件命名规范

一个好的文件名应该包含三类信息：**数据主题 + 粒度 + 时间范围**。

| 好的命名 | 差的命名 |
|----------|----------|
| `stock_daily_firmdate_2020_2026.parquet` | `data1.csv` |
| `fin_annual_firmyear_2020_2025.csv` | `new_data_final.xlsx` |
| `macro_shibor_monthly_2020_2026.csv` | `利率数据最新版.csv` |
| `analysis_base_firmyear_v1.csv` | `分析底表（用这个）.csv` |

文件名里带上时间范围，以后一眼就知道这份数据覆盖到什么时候，不需要打开文件查。

In [ ]:
# ── 生成数据字典 ─────────────────────────────────────
# 记录每张表的主键、粒度、字段、行数，方便后续查阅和交接

import glob

datasets = [
    ('stock_daily',   '个股日度行情',   'code + date',       'firm-date'),
    ('index_daily',   '市场指数日度',   'index_code + date', 'index-date'),
    ('fin_annual',    '年度财务指标',   'code + year',       'firm-year'),
    ('company_info',  '公司基本信息',   'code',              'firm'),
    ('shibor_3m',     'Shibor 3个月期', 'date',              'monthly'),
    ('usd_cny',       '人民币兑美元',   'date',              'monthly'),
    ('cpi_monthly',   'CPI 月度同比',   'date',              'monthly'),
]

lines = [f'# 数据字典\n\n生成时间：{pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}\n']

print(f'{"表名":<20} {"粒度":<12} {"主键":<20} {"行数":>6} 字段')
print('-' * 80)

for tbl, desc, pk, gran in datasets:
    try:
        df  = pd.read_sql(f'SELECT * FROM {tbl} LIMIT 1', conn)
        cnt = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tbl}', conn)['n'][0]
        cols = list(df.columns)
        print(f'{tbl:<20} {gran:<12} {pk:<20} {cnt:>6} {cols}')
        lines.append(f'\n## {tbl}\n- 描述：{desc}\n- 粒度：{gran}\n'
                     f'- 主键：{pk}\n- 行数：{cnt:,}\n- 字段：{cols}\n')
    except Exception as e:
        print(f'{tbl:<20} 读取失败：{e}')

dict_path = BASE / 'data_dictionary.md'
with open(dict_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)
print(f'\n数据字典已保存至 {dict_path}')

### 机构环境简介

本章介绍的工具链（SQLite + DuckDB + Parquet）适合个人分析和小团队项目。进入金融机构之后，你会遇到更大规模的数据基础设施，但底层逻辑是一样的：

- **MySQL / PostgreSQL**：服务器型关系数据库，支持多人并发访问。券商、基金的行情数据库和因子数据库通常用这类。你用的 SQL 语法 90% 可以直接迁移。
- **数据中台 / 数据仓库**：大型机构把各业务线数据统一接入一个平台，分析师通过 SQL 接口取数。
- **ClickHouse / TimescaleDB**：专门针对时间序列优化，处理 tick 数据（逐笔成交）时会用到。

这些工具现在不需要掌握，但了解它们的存在，能帮你在进入机构后更快适应数据环境。

---

## 本章小结

本章讨论的不是某个具体的分析方法，而是数据分析的「地基」——数据多了之后怎么管好它们。核心结论归纳为四点：

**1. 粒度和主键比语法更重要。** merge 之前先想清楚「每张表的每行代表什么」，先验证主键唯一性，能避免绝大多数数据合并错误。

**2. 格式选择取决于使用场景。** CSV 用于分享和查看，Parquet 用于大表的高效存储和读取，SQLite 用于多表关联查询，DuckDB 用于大数据量的分析型查询。

**3. SQL 是 pandas 的上游。** 用 SQL 做数据整合和过滤，用 pandas 做最终计算和可视化——这是实际项目中最常见的工作模式。

**4. 规范比工具更持久。** `data_raw/` 只写不改、文件名包含粒度和时间范围、维护下载日志和数据字典——这些习惯不需要额外工具，但能让你的项目在三个月后仍然可以被别人（以及未来的自己）理解和复用。

面对一个新的多来源数据项目，可以按这个顺序思考：

1. 列出所有原始数据表，写清楚每张表的粒度和主键
2. 确定最终分析底表的目标粒度
3. 判断哪些高频表需要先聚合再合并
4. 根据数据量和查询复杂度，选择存储格式
5. 建库、写 SQL 查询，用 `pd.read_sql()` 接入 pandas
6. 把分析底表持久化保存到 `data_clean/`，为后续清洗和建模做好准备